# Assignment — Introduction to NLP and Text Preprocessing

**Given text:**

> *"Natural Language Processing is an interesting field of Artificial Intelligence! It helps computers understand human language. NLP is used in chatbots, translation, and text analysis."*

## Setup

In [1]:
# If libraries are missing, run once (remove the # first):
# !pip install nltk spacy
# !python -m spacy download en_core_web_sm

import re
import string
import pandas as pd
import nltk
import spacy

for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg, quiet=True)

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    spacy.cli.download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

text = ("Natural Language Processing is an interesting field of Artificial Intelligence! "
        "It helps computers understand human language. "
        "NLP is used in chatbots, translation, and text analysis.")
print(text)

Natural Language Processing is an interesting field of Artificial Intelligence! It helps computers understand human language. NLP is used in chatbots, translation, and text analysis.


## Question 1: Syntax, Semantics, Pragmatics and Discourse

These are four levels of language analysis. Each one is applied to the given text below.

| Level | What it studies | Analysis of the given text |
|---|---|---|
| **Syntax** | Grammatical structure: word order, parts of speech, how words form phrases and sentences | Sentence 1 is *subject (Natural Language Processing) + linking verb (is) + noun phrase (an interesting field of Artificial Intelligence)*. Sentence 2 is *subject (It) + verb (helps) + object (computers) + verb phrase (understand human language)*. Sentence 3 is a **passive** construction (*NLP is used in …*) with a list of three nouns (*chatbots, translation, text analysis*). |
| **Semantics** | Literal meaning of words and sentences | The text says that NLP is a branch of AI, that its purpose is to let computers understand human language, and that it is applied in chatbots, translation and text analysis. Words such as *field* are ambiguous in general (farm field, field of study) but the context fixes the meaning as *area of study*. *NLP* and *Natural Language Processing* are the same concept. |
| **Pragmatics** | Meaning in context: the writer's intention and what the reader is expected to infer | The text is an **introductory, informative statement** meant to introduce NLP to a beginner. The exclamation mark shows enthusiasm. *Understand* is used loosely: the reader is expected to know that computers do not understand language like humans do. The reader is also expected to know what AI and chatbots are. |
| **Discourse** | How sentences connect to form a coherent text | The text moves from a **definition** (sentence 1), to its **purpose** (sentence 2), to its **applications** (sentence 3). *It* in sentence 2 refers back to *Natural Language Processing* (anaphora), and the abbreviation *NLP* in sentence 3 refers to the same concept (coreference). These links make the three sentences one coherent paragraph. |

**Code demonstration: syntax and discourse structure**

In [2]:
doc = nlp(text)

# Discourse: sentence segmentation
for i, sent in enumerate(doc.sents, 1):
    print(f"Sentence {i}: {sent.text}")

Sentence 1: Natural Language Processing is an interesting field of Artificial Intelligence!
Sentence 2: It helps computers understand human language.
Sentence 3: NLP is used in chatbots, translation, and text analysis.


In [3]:
# Syntax: part-of-speech tags and dependency relations (spaCy)
syntax = pd.DataFrame([
    {"Token": t.text, "POS": t.pos_, "Dependency": t.dep_, "Head": t.head.text}
    for t in doc if not t.is_space
])
syntax

,Token,POS,Dependency,Head
0,Natural,PROPN,compound,Language
1,Language,PROPN,compound,Processing
2,Processing,NOUN,nsubj,is
3,is,AUX,ROOT,is
4,an,DET,det,field
5,interesting,ADJ,amod,field
6,field,NOUN,attr,is
7,of,ADP,prep,field
8,Artificial,PROPN,compound,Intelligence
9,Intelligence,PROPN,pobj,of


In [4]:
# Semantics: the word "field" has many meanings; context selects one (WordNet)
from nltk.corpus import wordnet as wn
for syn in wn.synsets("field", pos=wn.NOUN)[:5]:
    print(f"{syn.name():<12} {syn.definition()}")

field.n.01   a piece of land cleared of trees and usually enclosed
battlefield.n.01 a region where a battle is being (or has been) fought
field.n.03   somewhere (away from a studio or office or library or laboratory) where practical work is done or data is collected
discipline.n.01 a branch of knowledge
field.n.05   the space around a radiating body within which its electromagnetic oscillations can exert force on another similar body not in contact with it


In [5]:
# Discourse: the pronoun "It" and the abbreviation "NLP" both point back to the same topic
print("Pronouns in text :", [t.text for t in doc if t.pos_ == "PRON"])
print("Acronyms in text :", re.findall(r"\b[A-Z]{2,}\b", text))

Pronouns in text : ['It']
Acronyms in text : ['NLP']


## Question 2: Two real-world applications of NLP

**1. Chatbots and virtual assistants** (customer-support bots, Siri, Alexa, Google Assistant)
Chatbots read the user's message and work out what the user wants (intent detection) and which details matter (entity extraction, e.g. a date or an order number). They then produce a reply. Preprocessing steps such as tokenization, stopword removal and lemmatization prepare the user's text for these models. Companies use chatbots to answer questions 24/7, which reduces waiting time and support cost.

**2. Machine translation** (Google Translate, DeepL)
Translation systems convert text from one language to another while keeping its meaning. They must handle grammar differences (syntax), word meaning (semantics) and context. Modern systems use neural networks trained on very large amounts of bilingual text. Translation lets people read websites, documents and conversations in languages they do not speak.

*Other examples:* sentiment analysis of product reviews, spam filtering, search engines, text summarization.

## Question 3: Possible noise in the text

**Noise** is any part of the text that does not help the NLP task and can make processing less accurate or less efficient. The given text is fairly clean, but it still contains these kinds of noise:

| Type of noise | Examples in the text |
|---|---|
| **Punctuation** | `!` `.` `,` |
| **Stopwords** (very common words with little meaning of their own) | *is, an, of, it, in, and* |
| **Inconsistent letter case** | *Natural*, *Language*, *Processing*, *Artificial*, *Intelligence* are capitalised, while the same words would appear in lower case elsewhere, so a computer would treat `Language` and `language` as different tokens |
| **Abbreviation and full form used together** | *NLP* and *Natural Language Processing* refer to the same concept but look different |
| **Different word forms** | *help / helps*, *use / used*, *understand* (inflections of the same base word) |

In [6]:
# Detect noise automatically
punct_found  = [c for c in text if c in string.punctuation]
upper_tokens = re.findall(r"\b[A-Z][a-zA-Z]*\b", text)
acronyms     = re.findall(r"\b[A-Z]{2,}\b", text)

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
stop_words = set(stopwords.words("english"))
stops_found = [w for w in word_tokenize(text.lower()) if w in stop_words]

print("Punctuation marks     :", punct_found, f"({len(punct_found)} total)")
print("Capitalised words     :", upper_tokens)
print("Acronyms              :", acronyms)
print("Stopwords found       :", stops_found)

Punctuation marks     : ['!', '.', ',', ',', '.'] (5 total)
Capitalised words     : ['Natural', 'Language', 'Processing', 'Artificial', 'Intelligence', 'It', 'NLP']
Acronyms              : ['NLP']
Stopwords found       : ['is', 'an', 'of', 'it', 'is', 'in', 'and']


## Question 4: Remove punctuation and stopwords

Steps: (1) convert to lower case, (2) tokenize, (3) remove punctuation, (4) remove stopwords.

In [7]:
# Step 1 and 2: lower case + tokenization
tokens = word_tokenize(text.lower())
print("Tokens           :", tokens)
print("Number of tokens :", len(tokens))

Tokens           : ['natural', 'language', 'processing', 'is', 'an', 'interesting', 'field', 'of', 'artificial', 'intelligence', '!', 'it', 'helps', 'computers', 'understand', 'human', 'language', '.', 'nlp', 'is', 'used', 'in', 'chatbots', ',', 'translation', ',', 'and', 'text', 'analysis', '.']
Number of tokens : 30


In [8]:
# Step 3: remove punctuation
no_punct = [t for t in tokens if t not in string.punctuation]
print("After removing punctuation :", no_punct)
print("Number of tokens           :", len(no_punct))

After removing punctuation : ['natural', 'language', 'processing', 'is', 'an', 'interesting', 'field', 'of', 'artificial', 'intelligence', 'it', 'helps', 'computers', 'understand', 'human', 'language', 'nlp', 'is', 'used', 'in', 'chatbots', 'translation', 'and', 'text', 'analysis']
Number of tokens           : 25


In [9]:
# Step 4: remove stopwords
clean_tokens = [t for t in no_punct if t not in stop_words]
print("After removing stopwords :", clean_tokens)
print("Number of tokens         :", len(clean_tokens))

print()
print("Cleaned text:", " ".join(clean_tokens))

After removing stopwords : ['natural', 'language', 'processing', 'interesting', 'field', 'artificial', 'intelligence', 'helps', 'computers', 'understand', 'human', 'language', 'nlp', 'used', 'chatbots', 'translation', 'text', 'analysis']
Number of tokens         : 18

Cleaned text: natural language processing interesting field artificial intelligence helps computers understand human language nlp used chatbots translation text analysis


In [10]:
# Summary: tokens removed at each step
removed_punct = [t for t in tokens if t in string.punctuation]
removed_stops = [t for t in no_punct if t in stop_words]
print("Punctuation removed :", removed_punct)
print("Stopwords removed   :", removed_stops)

Punctuation removed : ['!', '.', ',', ',', '.']
Stopwords removed   : ['is', 'an', 'of', 'it', 'is', 'in', 'and']


## Question 5: Text preprocessing with NLTK and spaCy

**NLTK (Natural Language Toolkit)** is a library for teaching and research. It provides many separate functions that the user combines step by step: `word_tokenize` / `sent_tokenize` (tokenization), `stopwords` (stopword lists), `PorterStemmer` (stemming), `WordNetLemmatizer` (lemmatization) and `pos_tag` (part-of-speech tagging). It gives full control over each step.

**spaCy** is an industrial-strength library. One call, `nlp(text)`, runs a complete pipeline and returns a `Doc` object whose tokens already have attributes such as `is_stop`, `is_punct`, `lemma_`, `pos_` and `dep_`. It is faster and easier to use for production work.

**NLTK example**

In [11]:
from nltk.tokenize import sent_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk import pos_tag

print("Sentences    :", sent_tokenize(text))
print()

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

# POS tags are assigned on the full sentence (context matters), then stopwords/punctuation are dropped
tagged = [(w.lower(), tag) for w, tag in pos_tag(word_tokenize(text))
          if w.lower() not in stop_words and w not in string.punctuation]

nltk_table = pd.DataFrame({
    "Token": [w for w, _ in tagged],
    "Stem (Porter)": [stemmer.stem(w) for w, _ in tagged],
    "Lemma (WordNet)": [lemmatizer.lemmatize(w) for w, _ in tagged],
    "POS tag": [tag for _, tag in tagged],
})
nltk_table

Sentences    : ['Natural Language Processing is an interesting field of Artificial Intelligence!', 'It helps computers understand human language.', 'NLP is used in chatbots, translation, and text analysis.']



,Token,Stem (Porter),Lemma (WordNet),POS tag
0,natural,natur,natural,JJ
1,language,languag,language,NNP
2,processing,process,processing,NNP
3,interesting,interest,interesting,JJ
4,field,field,field,NN
5,artificial,artifici,artificial,JJ
6,intelligence,intellig,intelligence,NN
7,helps,help,help,VBZ
8,computers,comput,computer,NNS
9,understand,understand,understand,VBP


**spaCy example**

In [12]:
doc = nlp(text)

# Complete preprocessing in one pass: lower case, remove punctuation and stopwords, lemmatize
spacy_tokens = [t.lemma_.lower() for t in doc if not t.is_punct and not t.is_stop and not t.is_space]
print("spaCy preprocessed tokens:", spacy_tokens)

spacy_table = pd.DataFrame([
    {"Token": t.text, "Lemma": t.lemma_, "POS": t.pos_, "Is stopword": t.is_stop, "Is punctuation": t.is_punct}
    for t in doc
])
spacy_table

spaCy preprocessed tokens: ['natural', 'language', 'processing', 'interesting', 'field', 'artificial', 'intelligence', 'help', 'computer', 'understand', 'human', 'language', 'nlp', 'chatbot', 'translation', 'text', 'analysis']


,Token,Lemma,POS,Is stopword,Is punctuation
0,Natural,Natural,PROPN,False,False
1,Language,Language,PROPN,False,False
2,Processing,processing,NOUN,False,False
3,is,be,AUX,True,False
4,an,an,DET,True,False
5,interesting,interesting,ADJ,False,False
6,field,field,NOUN,False,False
7,of,of,ADP,True,False
8,Artificial,Artificial,PROPN,False,False
9,Intelligence,Intelligence,PROPN,False,False


**Comparison**

| Feature | NLTK | spaCy |
|---|---|---|
| Design | Collection of separate tools, step by step | Single pipeline: `nlp(text)` does everything |
| Tokenization | `word_tokenize()` | `doc` tokens (automatic) |
| Stopwords | `stopwords.words('english')` list | `token.is_stop` |
| Punctuation | Filter with `string.punctuation` | `token.is_punct` |
| Stemming | Yes (`PorterStemmer`) | No (lemmatization only) |
| Lemmatization | `WordNetLemmatizer` (needs POS for best results) | `token.lemma_` (uses POS automatically) |
| Speed | Slower | Faster |
| Best for | Learning and research | Production applications |

## Question 6: Future directions of NLP

- **Large language models (LLMs):** Models trained on huge amounts of text will continue to improve at writing, reasoning, summarizing and answering questions, and are becoming part of everyday software.
- **Multilingual and low-resource languages:** Most NLP work focuses on English. Future work aims to support the many languages with little training data, such as Urdu and regional languages.
- **Multimodal NLP:** Systems that combine text with images, audio and video, for example describing an image or answering questions about a video.
- **Efficiency and on-device NLP:** Smaller, faster models that run on phones and laptops with lower cost, lower energy use and better privacy.
- **Fairness, safety and ethics:** Reducing bias, hallucinations (confident but false answers), misinformation and misuse of generated text.
- **Explainability and reliability:** Making it possible to understand why a model produced an answer, which matters in healthcare, law and education.
- **Domain-specific applications:** NLP tailored to medicine, law, finance and education, and more natural human–computer conversation.

## Conclusion

The text was analysed at the syntax, semantics, pragmatics and discourse levels. Its noise (punctuation, stopwords, letter case, abbreviation, word forms) was identified. After preprocessing, the 30 tokens produced by NLTK were reduced to 18 meaningful keywords. NLTK and spaCy were compared as preprocessing tools.